# 35 - Session 08: what dithering every fourth frame saves, and what it costs

**Purpose.** Run the analysis half of `protocols/08-dither-cadence.md` on `data/session08` and
publish `results/cadence_constants.json` and `results/cadence_frames.csv`:

1. **Download, settle and `t_dead`** with one dither every four frames. Session 06 dithered every
   frame, so `17` could only *infer* the split between saving a frame and settling after a dither.
   Tonight three frames in four have no dither in front of them, which measures the download
   directly.
2. **`eta_comb` with four frames per dither position**, against `25`'s every-frame ladder at matched
   stack size, and the protocol's decision rule applied to it.
3. **Star clipping per plane**, beside session 06's cell B on the same patch of sky.
4. **`F_sky` per plane** on a moonless evening - a by-product, recorded and not a headline.

This notebook is written for someone *checking* it. `36` reads its files back for someone deciding
what to do next.

**What the night delivered, and what it did not.** Checked from the headers, pixels and logs before
this notebook was written:

- Gates 1, 2 and 5 pass; the cloud test rejects nothing; 20 + 20 bias brackets.
- **The meridian flip came after frame 28.** The protocol gave the wrong transit time. Timing uses
  all 150 frames, because a rotation moves no gap. Everything that compares pixels uses the frames
  after the flip, as `17` did.
- **The ASIAIR refocused every two hours**, before frames 57 and 113. Rule 3 excludes those gaps.
- **The framing is off session 06's**, by about 200 px between the two nights' reference frames
  and more between others. The ROIs **follow the sky**: both boxes move by the plate-solved offset
  between the two references, so they cover the patch session 06 measured.
- 122 frames after the flip, not the 128 that two disjoint stacks of 64 would need. The ladder
  stops at the half-stack and says so.

**How the twin stacks are split - the one change to `25`'s method, and why it is forced.** A
stack's noise is read from the difference of two twin stacks of disjoint frames, and anything the
twins share cancels in that difference. `25` split alternate *frames* into the twins. Here that
would hide the very thing being measured: frames that share a dither position put the sensor's
fixed pattern on the same patch of sky, so if both twins hold frames from every position, the
pattern cancels between them and the ladder reports a stack that was dithered every frame. So the
twins are split by **dither position**. A takes every frame of the even positions and B every frame
of the odd ones. A stack of N then holds N/4 positions, as a real stack at this cadence does, and
its pattern does not cancel against its twin's. The positions alternate, so both twins still see
the same sky drift.

The same reasoning sets the **single-frame ideal**. `25` took it from consecutive frames, which
there always sat at different positions. Here consecutive frames mostly share one, so the ideal is
taken from **pairs at different positions** - frame j of one position against frame j of the next -
and it stays the quantity `25` measured. Pairs *within* a position are measured as well, as a
diagnostic: whatever is locked to a position cancels in their difference, so the gap between the
two is what a dither has to beat down.

**A same-night control.** One frame per position is a stack dithered every frame, drawn from
tonight. It gives a short ladder with no change of night, setpoint, focus or seeing, and it is
published beside the verdict as a check on rule 7's caveat. It decides nothing.

**Fixed before any number was seen.**

- `25`'s stacking settings, unchanged: Lanczos-3 registration of each Bayer plane on its own,
  Winsorized sigma clip 4.0 / 3.0 plus a no-rejection arm, average, additive normalisation, equal
  weights, noise at 4x4 binning from differences.
- **The reference** is the post-flip frame nearest the middle of the post-flip night. Its whole
  dither position is left out of every stack, so every position in the ladder holds exactly four
  frames; a position with fewer than four usable frames is left out for the same reason.
- **Rungs** 3, 4, 8, 16, 18 and 32, plus the half-stack. Matched against `25` at 3, 4, 8, 16 and 18,
  the rungs it has.
- **The decision rule is the protocol's**: keep every-fourth-frame dithering if
  `eta_comb(4 per position) / eta_comb(every frame)` stays at or above the threshold **at every
  matched rung**. The threshold is 0.92 if the download comes out fast and 0.94 if slow, and
  "fast" means nearer the archive's 0.68 s than session 06's 7.99 s bound.
- **`t_dead`** is published two ways. Rule 4's, as written: the mean gap over the night with
  interruptions left out. And **`download + settle / 4`**, the cost of a clean night at this
  cadence, which is the one the model uses (agreed with Denis, 2026-10-06). They differ because the
  three interruption gaps each also hold a dither, so leaving them out leaves the night slightly
  under one dither in four.
- **Star clipping** by `29`'s method on tonight's frames: a deep image of every usable post-flip
  sub, isolated stars by `spatial.stars`, each star's peak the median over subs of its brightest
  pixel within 1 px, clipped if that median is 4095, and counted again against
  `linear_to_at_least`. **Only stars on the patch of sky both nights cover, on both lists**, with a
  border of 64 plane pixels.

**What it is not for.** Not a gain comparison, a ranked pair or the SNR estimator's repeatability.
Not guiding quality: the guide log is kept, but gate 3 stays the cloud test, because a guiding
threshold chosen after seeing the night would be fitted to it. Not the mount's limit on long subs.
Not N = 64. Not whether the model *should* switch cadence - that is `36`'s question, and a
conversation after it. And not a picture.

**It assumes** `18_sky_pair_read.ipynb` for the sky night, `26_contract3_read.ipynb` for the ladder
and why noise is read from differences at 4x4, `30_star_clipping_read.ipynb` for the star method, and
`34_dither_flip_read.ipynb` for why the cadence matters.

In [ ]:
import json
import pathlib
import re
import shutil
import sys
import time

import numpy as np
import pandas as pd
from astropy.io import fits as afits
from astropy.wcs import WCS

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, model as M, pixinsight as PI, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session08"
LIGHTS = DATA / "Light" / "NGC 7000"
DATA06 = ROOT / "data" / "session06"
# Intermediates are disposable and live on the NAS (CLAUDE.md).  Cells below
# skip work whose result file is already here, so a re-run after a crash does
# not redo the registration; delete the folder to start clean.
TEMP = pathlib.Path(r"Z:\pix\_astro\astropix\temp\contract3\cadence")
for sub in ("split", "reg", "ref", "stack"):
    (TEMP / sub).mkdir(parents=True, exist_ok=True)

sky06 = json.loads((RESULTS / "sky_constants.json").read_text())
stack06 = json.loads((RESULTS / "stack_constants.json").read_text())
mc = json.loads((RESULTS / "model_constants.json").read_text())
bias_sweep = pd.read_csv(RESULTS / "bias_sweep.csv")

GAIN, OFFSET, EXPTIME, SET_TEMP = 50, 15, 120.0, -10.0
CADENCE = 4                       # frames per dither, as the ASIAIR was set
BARE_FACTOR = 3.0                 # a gap over 3x the night's median is not a bare download
INTERRUPTION_FACTOR = 3.0         # rule 3: over 3x the median dither-frame gap
REJECT_SIGMAS = 4.0               # gate 3, session 06's cloud test, unchanged
TEMP_BAND = 0.5                   # C; a frame outside it is flagged, not dropped
G = mc["system_gain"]["value"][str(GAIN)]                  # e- per ADC count, -10 C
CEILING = mc["linear_to_at_least"]["value"][str(GAIN)]     # ADC counts
FULL_SCALE = stats.ADC_FULL_SCALE
DECOMP06 = sky06["t_dead_decomposition"]["value"]
DOWNLOAD_PREDICTED = {"fast": DECOMP06["download_and_save_archive"],
                      "slow": DECOMP06["download_and_save_bound"]}
THRESHOLD = {"fast": 0.92, "slow": 0.94}                   # the protocol's, before the data

K = stats.NOISE_BIN
SCALES = (1, K)
PLANES = spatial.PLANES
RUNGS = [3, 4, 8, 16, 18, 32]
MATCHED = [3, 4, 8, 16, 18]       # the rungs 25's ladder has
CONTROL_RUNGS = [3, 4, 8]
REJECTION = {"none": {"rejection": "none"},
             "winsorized": {"rejection": "winsorized", "sigma_low": 4.0, "sigma_high": 3.0}}
COMMON = {"combination": "average", "normalization": "additive", "weight_mode": "dont_care"}
CHUNK = 30                        # frames per registration launch, so a crash costs one chunk

TILES, RADIUS = (4, 4), 1         # 29's star method
DETECT = {"nsigma": 5.0, "background": 31, "isolation": 5}
BORDER = 64                       # plane px kept clear of both nights' frame edges

FRAMES_CSV = RESULTS / "cadence_frames.csv"
CONSTANTS = RESULTS / "cadence_constants.json"

probe = PI.run(PI.scripts_dir() / "probe.js", {"an_int": 42})
assert probe["ok"], probe.get("error")
print(f"PixInsight {probe['core']['version']} build {probe['core']['build']}")
print(f"g {G} e-/count, linear_to_at_least {CEILING:.1f} counts at gain {GAIN}")

## 1. The night, from the headers

Every light's capture settings, read from the header and trusted: the protocol set them. Then the
flip, decided by `17`'s rule, stated before the count is looked at: **the night's modal camera angle
is the night, and every frame at a different angle is out of the pixel work.**

In [ ]:
night, HEADERS = [], {}
for path in sorted(LIGHTS.glob("*.fit")):
    h = afits.getheader(path)
    HEADERS[path.name] = h
    night.append(F.capture_settings(h) | {"file": path.name, "rotator": h.get("ROTATOR"),
                                         "focuspos": h.get("FOCUSPOS"), "solved": "CD1_1" in h})
night = pd.DataFrame(night)
night["t"] = pd.to_datetime(night.date_obs)
night = night.sort_values("t").reset_index(drop=True)
night["n"] = np.arange(1, len(night) + 1)

settings = night[["gain", "offset", "exptime", "set_temp"]].drop_duplicates()
assert len(settings) == 1 and tuple(settings.iloc[0]) == (GAIN, OFFSET, EXPTIME, SET_TEMP), settings
night["temp_off"] = night.ccd_temp - night.set_temp
night["temp_flag"] = night.temp_off.abs() > TEMP_BAND
print(f"{len(night)} lights, gain {GAIN}, offset {OFFSET}, {EXPTIME:.0f} s, set {SET_TEMP:.0f} C; "
      f"{night.t.iloc[0]:%H:%M} to {night.t.iloc[-1]:%H:%M} UTC")
print(f"CCD-TEMP {night.ccd_temp.min()} to {night.ccd_temp.max()} C; "
      f"{int(night.temp_flag.sum())} frames outside +-{TEMP_BAND} C")
print(f"unsolved headers: {night.n[~night.solved].tolist()}  (registration matches stars, "
      "not headers)")

MODAL_ANGLE = float(night.rotator.mode().iloc[0])
night["flipped"] = (night.rotator - MODAL_ANGLE).abs() > 1.0
print(f"\ncamera angles {sorted(night.rotator.unique())}; modal {MODAL_ANGLE:.0f} deg -- the night")
print(f"{int(night.flipped.sum())} frames before the flip (frames "
      f"{night.n[night.flipped].min()}-{night.n[night.flipped].max()}), out of the pixel work")
print("focus positions:", night.groupby("focuspos").n.agg(["min", "max", "size"])
      .rename(columns={"min": "first", "max": "last"}).to_dict("index"))

## 2. The gaps: download, settle and `t_dead`

The gap *in front of* a frame is its start minus the previous frame's start minus the exposure.
A gap is a **bare download** unless it is more than three times the night's median gap: three gaps
in four are bare downloads, so the median is one. The rest followed a dither, and which frames they
are is checked two ways - against the cadence the ASIAIR was set to (a dither before frames 5, 9,
13, ...), and against the ASIAIR's own log, which names every dither and times every settle. A
disagreement is reported, not smoothed.

**Rule 3**: a gap is an interruption - flip, autofocus - if it is over 3x the *median dither-frame
gap*. **Rule 4**: download is the mean bare gap; settle is the mean dither gap minus the download;
`t_dead` is the mean over every gap, interruptions excluded. Means, never medians.

In [ ]:
night["gap_s"] = night.t.diff().dt.total_seconds() - night.exptime.shift()
median_gap = float(night.gap_s.median())
night["dither"] = night.gap_s > BARE_FACTOR * median_gap
expected = (night.n > 1) & ((night.n - 1) % CADENCE == 0)
off_cadence = night.n[(night.dither != expected) & (night.n > 1)].tolist()
print(f"median gap {median_gap:.3f} s; {int(night.dither.sum())} dither gaps; "
      f"against every {CADENCE} frames: {off_cadence or 'no disagreement'}")

# The ASIAIR log: the k-th 120 s exposure is light k; a dither between two
# exposures belongs to the second.  Only the night's own run is read -- the
# file goes on to log a short run the next morning, which has no 120 s frames.
LOG = next(p for p in DATA.glob("Autorun_Log_*.txt") if "_CHN" not in p.name)
light, dither_at, settles = 0, None, []
log_dither = np.zeros(len(night), bool)
for line in LOG.read_text(encoding="utf-8").splitlines():
    m = re.match(r"(\d{4}/\d\d/\d\d \d\d:\d\d:\d\d) (.*)", line)
    if not m:
        continue
    t, event = pd.Timestamp(m[1].replace("/", "-")), m[2].strip()
    if event.startswith(f"Exposure {EXPTIME:.1f}s"):
        light += 1
    elif event == "[Guide] Dither":
        dither_at = t
        if light < len(night):
            log_dither[light] = True          # index `light` is the next light, 0-based
    elif event in ("[Guide] Settle Done", "[Guide] Settle Timeout") and dither_at is not None:
        settles.append({"before_light": light + 1, "settle_s": (t - dither_at).total_seconds(),
                        "timeout": event.endswith("Timeout")})
        dither_at = None
    if event.startswith("[Autorun|End]"):
        break
assert light == len(night), f"the log counts {light} lights, the folder holds {len(night)}"
night["dither_log"] = log_dither
settles = pd.DataFrame(settles)
settles = settles[settles.before_light > 1]              # the one before light 1 is not a gap
disagree = night.n[(night.dither != night.dither_log) & (night.n > 1)].tolist()
print(f"the log names {int(night.dither_log[night.n > 1].sum())} dithers between lights; "
      f"against the gaps: {disagree or 'no disagreement'}")
print(f"settle timeouts in the log: before light {settles.before_light[settles.timeout].tolist()}")

In [ ]:
dith_median = float(night.gap_s[night.dither].median())
night["interruption"] = night.gap_s > INTERRUPTION_FACTOR * dith_median
print(f"median dither gap {dith_median:.2f} s; interruption cut {INTERRUPTION_FACTOR * dith_median:.1f} s")
print(night[night.interruption][["n", "t", "gap_s", "rotator", "focuspos"]]
      .round({"gap_s": 1}).to_string(index=False))

steady = night[night.gap_s.notna() & ~night.interruption]
bare, dith = steady.gap_s[~steady.dither], steady.gap_s[steady.dither]


def se(x):
    return float(x.std(ddof=1) / np.sqrt(len(x)))


download, download_se = float(bare.mean()), se(bare)
settle = float(dith.mean()) - download
settle_se = float(np.hypot(se(dith), download_se))
t_dead_rule4, t_dead_rule4_se = float(steady.gap_s.mean()), se(steady.gap_s)
t_dead = download + settle / CADENCE
t_dead_se = float(np.hypot(download_se, settle_se / CADENCE))
t_dead_every = download + settle                  # the same rig, dithering every frame
dither_share = float(steady.dither.mean())
settle_log = float(settles.settle_s.mean())

print(f"\ndownload          {download:6.3f} +- {download_se:.3f} s   over {len(bare)} bare gaps "
      f"(range {bare.min():.2f}-{bare.max():.2f})")
print(f"settle            {settle:6.2f} +- {settle_se:.2f} s    over {len(dith)} dither gaps; the log "
      f"says {settle_log:.2f} s over {len(settles)} (1 s resolution, interruptions included)")
print(f"t_dead, rule 4    {t_dead_rule4:6.2f} +- {t_dead_rule4_se:.2f} s    over {len(steady)} gaps, "
      f"{dither_share:.3f} of them dithers")
print(f"t_dead, cadence   {t_dead:6.2f} +- {t_dead_se:.2f} s    download + settle / {CADENCE}  "
      "<- what the model uses")
print(f"   identity check: download + settle x {dither_share:.3f} = "
      f"{download + settle * dither_share:.2f} s, against rule 4's {t_dead_rule4:.2f} s")

REGIME = min(DOWNLOAD_PREDICTED, key=lambda k: abs(DOWNLOAD_PREDICTED[k] - download))
THRESH = THRESHOLD[REGIME]
snr_gain = float(np.sqrt((EXPTIME + t_dead_every) / (EXPTIME + t_dead)))
print(f"\nthe download is {REGIME} (predicted {DOWNLOAD_PREDICTED}); the decision threshold "
      f"is therefore {THRESH}")
print(f"if stacking loses nothing, every-4th-frame dithering buys {100 * (snr_gain - 1):.1f}% SNR at "
      f"{EXPTIME:.0f} s against every-frame dithering on this rig ({t_dead_every:.1f} s); the "
      f"break-even efficiency ratio is {1 / snr_gain:.4f}, beside the protocol's {THRESH}")

## 3. The reference, and the boxes moved onto session 06's sky

The reference is the post-flip frame nearest the middle of the post-flip night. Session 06's is
`25`'s rule, re-derived from `sky_frames.csv`. Each ROI's centre goes from session 06's reference
pixels to the sky through that frame's plate solve, and from the sky to tonight's reference pixels
through tonight's, then onto even coordinates so a box starts on an R pixel. The two nights' fields
also differ by a small turn, which a shift cannot follow; how far it moves a box's corners is printed.

In [ ]:
post = night[~night.flipped]
mid = post.t.min() + (post.t.max() - post.t.min()) / 2
REF = post.loc[(post.t - mid).abs().idxmin()]
assert REF.solved

f06 = pd.read_csv(RESULTS / "sky_frames.csv", parse_dates=["t"])
g06 = f06[~f06.flipped & ~f06.rejected]
mid06 = g06.t.min() + (g06.t.max() - g06.t.min()) / 2
d06 = g06[g06.cell == "D"]
REF06 = d06.loc[(d06.t - mid06).abs().idxmin()]
h06, h08 = afits.getheader(DATA06 / REF06.file), HEADERS[REF.file]
w06, w08 = WCS(h06), WCS(h08)
print(f"reference tonight: frame {REF.n}, {REF.file}")
print(f"reference session 06 (25's rule): {REF06.file}")


def follow(roi):
    x, y, w, h = roi
    ra, dec = w06.all_pix2world(x + w / 2, y + h / 2, 0)
    cx, cy = w08.all_world2pix(ra, dec, 0)
    return (int(round((cx - w / 2) / 2)) * 2, int(round((cy - h / 2) / 2)) * 2, w, h)


ROI06 = sky06["sky_roi"]["value"]
SIGNAL_MOSAIC, SKY_MOSAIC = follow(ROI06["signal_roi"]), follow(ROI06["roi"])
for name, old, new in (("signal", ROI06["signal_roi"], SIGNAL_MOSAIC), ("sky", ROI06["roi"], SKY_MOSAIC)):
    x, y, w, h = new
    assert x >= BORDER and y >= BORDER and x + w <= 3840 - BORDER and y + h <= 2160 - BORDER, new
    print(f"{name} ROI: session 06 {tuple(old)} -> tonight {new}, moved "
          f"{np.hypot(new[0] - old[0], new[1] - old[1]):.0f} px")
SIGNAL_ROI, SKY_ROI = spatial.plane_roi(SIGNAL_MOSAIC), spatial.plane_roi(SKY_MOSAIC)


def angle(h):
    return np.degrees(np.arctan2(h["CD2_1"], h["CD1_1"]))


TURN = float((angle(h08) - angle(h06) + 180) % 360 - 180)
half_diag = np.hypot(*SIGNAL_MOSAIC[2:]) / 2
print(f"field turn between the nights {TURN:+.2f} deg: moves the signal box's corners "
      f"{np.radians(abs(TURN)) * half_diag:.1f} mosaic px against its centre")
print(f"plane ROIs: signal {SIGNAL_ROI}, sky {SKY_ROI}")

## 4. Every frame's sky, and the cloud test

The sky level is `stats.sky_level` - the mode of the sky ROI, per plane, on the raw sub - exactly as
`17` measured it. Post-flip frames only: before the flip the box is a different patch of sky. The
cloud test is gate 3, `17`'s rule unchanged: a frame is rejected if its green sky mode departs from
the night's median by more than 4.0 MADs.

In [ ]:
def planes_of(name):
    return spatial.split(stats.to_adc(F.read(LIGHTS / name)[0]))


t0 = time.time()
for p in PLANES:
    night[f"sky_mode_{p}"] = np.nan
for i, r in night[~night.flipped].iterrows():
    pl = planes_of(r.file)
    for p in PLANES:
        night.loc[i, f"sky_mode_{p}"] = stats.sky_level(spatial.cut(pl[p], SKY_ROI))
night["sky_mode_green"] = night[["sky_mode_G1", "sky_mode_G2"]].mean(axis=1)
print(f"sky modes in {time.time() - t0:.0f} s")

post = night[~night.flipped]
sky_med = float(post.sky_mode_green.median())
sky_mad = stats.MAD_TO_SIGMA * float(np.median(np.abs(post.sky_mode_green - sky_med)))
night["sky_dev"] = (night.sky_mode_green - sky_med) / sky_mad
night["rejected"] = night.sky_dev.abs() > REJECT_SIGMAS
print(f"green sky mode: median {sky_med:.2f} counts, MAD-sigma {sky_mad:.2f}; "
      f"largest departure {night.sky_dev.abs().max():.2f} MADs at frame "
      f"{int(night.n[night.sky_dev.abs().idxmax()])}")
print(f"rejected: {night.n[night.rejected].tolist() or 'none'}")
good = night[~night.flipped & ~night.rejected].copy()
print(f"{len(good)} usable frames after the flip; the protocol's floor is 64")
assert len(good) >= 64, "fewer than 64 usable frames is a session that did not run -- reshoot"

## 5. The brackets: gate 1, and the pedestal

Gate 1 is run again here, on every bias frame rather than the one pulled at the mount: the modal
step between adjacent values must be 16 on all four planes, in **stored** units - one of the four
places the project keeps them, because in ADC counts the test is vacuous. Then the pedestal (rule
2): the offset state of each bracket frame is assigned with `stats.offset_state`, and the pedestal
is the per-plane mean of the frames in the populated state.

In [ ]:
brackets = []
for path in sorted((DATA / "Bias").glob("*.fit")):
    mosaic, h = F.read(path)
    stored, adc = spatial.split(mosaic), spatial.split(stats.to_adc(mosaic))
    brackets.append({"file": path.name, "t": pd.Timestamp(h["DATE-OBS"]), "gain": h["GAIN"],
                     "offset": h["OFFSET"], "exptime": h["EXPTIME"], "ccd_temp": h["CCD-TEMP"],
                     **{f"step_{p}": stats.value_step(stored[p]) for p in PLANES},
                     **{f"mean_{p}": float(adc[p].mean()) for p in PLANES}})
brackets = pd.DataFrame(brackets)
assert (brackets.gain == GAIN).all() and (brackets.offset == OFFSET).all()
steps = brackets[[f"step_{p}" for p in PLANES]]
GATE1 = bool((steps == 16).all().all())
print(f"gate 1: modal step {sorted(set(steps.to_numpy().ravel()))} over {len(brackets)} frames "
      f"x 4 planes -> {'PASS' if GATE1 else 'FAIL'}")
assert GATE1, "white balance is still applied; nothing from this night is usable"

brackets["bracket"] = np.where(brackets.t < night.t.min(), "start", "end")
brackets["level"] = brackets[[f"mean_{p}" for p in PLANES]].mean(axis=1)
state = stats.offset_state(brackets.level)
near = ~state["far"]
brackets["far"] = state["far"]
STATE_SEP = state["separation"] or 0.0
print(f"offset states: centres {np.round(state['centres'], 3).tolist()}, separation "
      f"{STATE_SEP:.3f} counts, {int(state['far'].sum())} frames far")
print(brackets.groupby("bracket")[["level", "far"]].agg({"level": ["size", "mean", "std"],
                                                        "far": "sum"}).round(4).to_string())

PEDESTAL = {p: float(brackets.loc[near, f"mean_{p}"].mean()) for p in PLANES}
PEDESTAL_SD = float(brackets.loc[near, "level"].std(ddof=1))
# A light cannot be put in a state on its own -- its sky is far larger than
# the step -- so its pedestal is uncertain by half the step as well as by the
# scatter inside the populated state.
PEDESTAL_U = float(np.hypot(PEDESTAL_SD, STATE_SEP / 2))
published = float(bias_sweep.query("gain == @GAIN and offset == @OFFSET").pedestal.iloc[0])
print(f"pedestal per plane {({p: round(v, 3) for p, v in PEDESTAL.items()})}; sd in the populated "
      f"state {PEDESTAL_SD:.3f}, with half the state step {PEDESTAL_U:.3f}; bias_sweep.csv at "
      f"gain {GAIN}, offset {OFFSET}: {published:.3f}")

## 6. `F_sky` per plane - the by-product

Rule 3 of session 06, unchanged: the modal sky level minus the pedestal, times `g`, over the
exposure, per frame, averaged over the usable frames. `g` is the -10 C bench value, which is where
tonight was shot. Laid beside session 06's cell B - the same setting, a different night, and -20 C
then, though at this sky level the dark term is negligible at both.

In [ ]:
for p in PLANES:
    night[f"F_sky_{p}"] = (night[f"sky_mode_{p}"] - PEDESTAL[p]) * G / night.exptime
night["F_sky_green"] = night[["F_sky_G1", "F_sky_G2"]].mean(axis=1)
good = night[~night.flipped & ~night.rejected].copy()
F_SKY = {p: float(good[f"F_sky_{p}"].mean()) for p in PLANES + ("green",)}
F_SKY_SD = {p: float(good[f"F_sky_{p}"].std(ddof=1)) for p in PLANES + ("green",)}
F_SKY_PED_ERR = PEDESTAL_U * G / EXPTIME
cell_b = sky06["F_sky_per_cell"]["value"]["B"]
print(pd.DataFrame({"tonight": F_SKY, "night sd": F_SKY_SD,
                    "session 06 cell B": cell_b | {"green": (cell_b["G1"] + cell_b["G2"]) / 2}})
      .round(4).to_string())
print(f"pedestal uncertainty per frame: {F_SKY_PED_ERR:.4f} e-/px/s")

## 7. The ladder's frames, and registration

Dither positions are counted from the gaps: a new one starts at every dither. The reference's
position is left out, and so is any position without exactly four usable frames. A takes the even
positions and B the odd ones, in time order. About 4 s a frame per plane in `register.js`; a chunk
already registered is reused.

In [ ]:
night["position"] = night.dither.cumsum()
good = night[~night.flipped & ~night.rejected].copy()
size = good.groupby("position").size()
REF_POS = int(night.loc[night.file == REF.file, "position"].iloc[0])
LADDER_POS = [int(p) for p, s in size.items() if s == CADENCE and p != REF_POS]
left_out = {int(p): int(s) for p, s in size.items() if int(p) not in LADDER_POS}
A_POS, B_POS = LADDER_POS[0::2], LADDER_POS[1::2]
frames_at = {p: good[good.position == p].sort_values("t").n.tolist() for p in LADDER_POS}
in_ladder = good[good.position.isin(LADDER_POS)].sort_values("t")
night["twin"] = np.select([night.position.isin(A_POS) & night.n.isin(in_ladder.n),
                           night.position.isin(B_POS) & night.n.isin(in_ladder.n),
                           night.file == REF.file], ["A", "B", "reference"], "")
print(f"{len(LADDER_POS)} positions of {CADENCE} in the ladder ({len(in_ladder)} frames): "
      f"A {len(A_POS)}, B {len(B_POS)}")
print(f"left out, position: usable frames -> {left_out}  (the reference's is {REF_POS})")


def stem(n):
    return f"s08_{n:03d}"


reg = []
for c, i in enumerate(range(0, len(in_ladder), CHUNK)):
    out = TEMP / f"register_{c}.json"
    if not out.exists():
        rows = in_ladder.iloc[i:i + CHUNK]
        job = {"reference": PI.pi_path(LIGHTS / REF.file), "outdir": PI.pi_path(TEMP),
               "interpolation": "lanczos3",
               "frames": [{"path": PI.pi_path(LIGHTS / f), "stem": stem(n)}
                          for f, n in zip(rows.file, rows.n)]}
        t0 = time.time()
        r = PI.run(PI.scripts_dir() / "register.js", job, timeout=3600)
        assert r["ok"], r.get("error")
        r.pop("log", None)
        out.write_text(json.dumps(r, indent=1))
        print(f"chunk {c}: {len(rows)} frames registered in {time.time() - t0:.0f} s")
    reg.append(json.loads(out.read_text()))
print("settings:", reg[0]["settings"])

In [ ]:
qual = pd.DataFrame([
    {"stem": f["stem"], "plane": p, "ok": v["ok"], "path": v.get("output", ""),
     "matches": v.get("matches"), "rms_error": v.get("rms_error"),
     "dx": v.get("H13"), "dy": v.get("H23")}
    for r in reg for f in r["frames"] for p, v in f["planes"].items()])
assert set(qual.stem) == {stem(n) for n in in_ladder.n}, "registered set is not the ladder's"
print(f"{(~qual.ok).sum()} of {len(qual)} frame-planes failed to register")
print(qual.groupby("plane").agg(matches=("matches", "min"), rms_worst=("rms_error", "max"))
      .round(3).to_string())
assert qual.ok.all(), "a frame that did not register cannot be stacked; see the failures above"
print(f"\ndither + drift against the reference: |dx| up to {qual.dx.abs().max():.1f}, "
      f"|dy| up to {qual.dy.abs().max():.1f} plane pixels")
path_of = qual.set_index(["stem", "plane"]).path

## 8. Single frames: the ideal, and what is locked to a position

**Across positions** - frame j of position 2k against frame j of position 2k+1 - gives the ideal
`25` used: everything that differs from frame to frame. **Within a position** - frames (0, 1) and
(2, 3) of one position - anything locked to the position cancels. So

`locked = sqrt(across^2 - within^2) / across`

is the share of a single frame's noise that a stack only beats down by changing position.

**What it is made of is not settled here.** The sensor's fixed pattern is locked to a position. So
is the registration: frames at one position are resampled with nearly the same sub-pixel shift, so
the kernel's error on a star is nearly the same in both and cancels, where across positions it does
not. And within-position pairs are 2 minutes apart against about 9, so slow seeing and sky changes
cancel more within. The ladder measures with all of it in, as `25` did, so the comparison stays
fair. As a pointer to how much is stars, the same numbers are given with `27`'s tight star mask
(5 sigma, grown 3 px, on the mean of every frame) taken out of the 4x4 difference.

In [ ]:
across = [(frames_at[a][j], frames_at[b][j]) for a, b in zip(A_POS, B_POS) for j in range(CADENCE)]
within = [(fs[j], fs[j + 1]) for fs in frames_at.values() for j in range(0, CADENCE, 2)]
MASK_KW = {"nsigma": 5.0, "grow": 3}          # 27's 'tight' mask
t0 = time.time()
singles, MASK_SHARE = [], {}
for p in PLANES:
    cut = {n: spatial.cut(PI.read_adc(path_of[stem(n), p]), SIGNAL_ROI).astype(np.float32)
           for n in in_ladder.n}
    assert min(c.min() for c in cut.values()) > 0, "a dither border reaches into the signal ROI"
    mask = spatial.star_mask(np.mean(list(cut.values()), axis=0), **MASK_KW)
    MASK_SHARE[p] = float(mask.mean())
    for kind, pairs in (("across", across), ("within", within)):
        for a, b in pairs:
            singles.append({"kind": kind, "plane": p, "a": a, "b": b,
                            **{f"sigma_{k}": stats.diff_sigma(cut[a], cut[b], k) for k in SCALES},
                            "sigma_masked": stats.diff_sigma(cut[a], cut[b], K, mask=mask)})
singles = pd.DataFrame(singles)
print(f"{len(across)} pairs across positions, {len(within)} within, in {time.time() - t0:.0f} s; "
      f"the star mask covers {({p: round(v, 3) for p, v in MASK_SHARE.items()})} of the ROI")

one = singles.pivot_table(index="plane", columns="kind",
                          values=[f"sigma_{k}" for k in SCALES] + ["sigma_masked"])
for col in [f"sigma_{k}" for k in SCALES] + ["sigma_masked"]:
    a, w = one[(col, "across")], one[(col, "within")]
    one[(col.replace("sigma", "locked"), "")] = np.sqrt(np.clip(a ** 2 - w ** 2, 0, None)) / a
IDEAL = {p: {k: float(one.loc[p, (f"sigma_{k}", "across")]) for k in SCALES} for p in PLANES}
print(one.round(4).to_string())

## 9. The stacks

Per plane, one launch each:

| label | frames | arms |
|---|---|---|
| `A{N}`, `B{N}` for N in 3, 4, 8, 16, 18, 32 | the first N frames of the even / odd positions | none, winsorized |
| `hA`, `hB` | all of them, trimmed to equal length | none, winsorized |
| `oA{N}`, `oB{N}` for N in 3, 4, 8 and the longest | **one** frame per position - the control | winsorized |

The control takes frame `i mod 4` of its i-th position, so it draws on every slot after a dither and
not only the first.

In [ ]:
def frames_of(positions):
    return [n for p in positions for n in frames_at[p]]


A, B = frames_of(A_POS), frames_of(B_POS)
HALF = min(len(A), len(B))
assert max(RUNGS) <= HALF, f"the half-stack is {HALF}, under the top rung {max(RUNGS)}"
oA = [frames_at[p][i % CADENCE] for i, p in enumerate(A_POS)]
oB = [frames_at[p][i % CADENCE] for i, p in enumerate(B_POS)]
CONTROL_TOP = min(len(oA), len(oB))
main = ({f"A{n}": A[:n] for n in RUNGS} | {f"B{n}": B[:n] for n in RUNGS}
        | {"hA": A[:HALF], "hB": B[:HALF]})
control = ({f"oA{n}": oA[:n] for n in CONTROL_RUNGS + [CONTROL_TOP]}
           | {f"oB{n}": oB[:n] for n in CONTROL_RUNGS + [CONTROL_TOP]})
print(f"half-stack {HALF} frames ({HALF // CADENCE} positions); control reaches {CONTROL_TOP}")

stacks = {}
for p in PLANES:
    out = TEMP / f"integrate_{p}.json"
    runs = []
    for arm, groups in (("none", main), ("winsorized", main | control)):
        for label, ns in groups.items():
            f = TEMP / "stack" / f"{p}_{arm}_{label}.fits"
            runs.append({"label": f"{p}|{arm}|{label}", "output": PI.pi_path(f),
                         "frames": [PI.pi_path(path_of[stem(n), p]) for n in ns],
                         **COMMON, **REJECTION[arm]})
    if not out.exists():
        t0 = time.time()
        r = PI.run(PI.scripts_dir() / "integrate.js", {"runs": runs}, timeout=5400)
        assert r["ok"], r.get("error")
        r.pop("log", None)
        out.write_text(json.dumps(r, indent=1))
        print(f"plane {p}: {len(runs)} integrations in {time.time() - t0:.0f} s")
    r = json.loads(out.read_text())
    bad = [x["label"] for x in r["runs"] if not x["ok"]]
    assert not bad, f"integrations that failed: {bad}"
    for x in r["runs"]:
        stacks[x["label"]] = {"n": x["n"], "path": x["output"], "settings": x["settings"]}
print(f"{len(stacks)} stacks; settings read back, e.g.:", next(iter(stacks.values()))["settings"])

## 10. The ladder

`sigma_N` is one stack's noise, from the difference of its twins. `eta_comb = (ideal / sqrt(N)) /
sigma_N`, with the across-position ideal of section 8 - the mean over every pair, never a frame that
is also in the stack.

In [ ]:
def load(p, arm, label):
    return spatial.cut(PI.read_adc(stacks[f"{p}|{arm}|{label}"]["path"]), SIGNAL_ROI)


pairs = ([("four", "none", f"A{n}", f"B{n}") for n in RUNGS] + [("four", "none", "hA", "hB")]
         + [("four", "winsorized", f"A{n}", f"B{n}") for n in RUNGS]
         + [("four", "winsorized", "hA", "hB")]
         + [("one", "winsorized", f"oA{n}", f"oB{n}") for n in CONTROL_RUNGS + [CONTROL_TOP]])
ladder = []
for p in PLANES:
    for kind, arm, a, b in dict.fromkeys(pairs):
        n = stacks[f"{p}|{arm}|{a}"]["n"]
        assert n == stacks[f"{p}|{arm}|{b}"]["n"]
        sa, sb = load(p, arm, a), load(p, arm, b)
        row = {"plane": p, "per_position": CADENCE if kind == "four" else 1, "arm": arm, "n": n,
               "positions": -(-n // CADENCE) if kind == "four" else n}
        for k in SCALES:
            row[f"sigma_{k}"] = stats.diff_sigma(sa, sb, k)
            row[f"eta_comb_{k}"] = PI.eta_comb(IDEAL[p][k], row[f"sigma_{k}"], n)
        ladder.append(row)
ladder = pd.DataFrame(ladder)
view = ladder.pivot_table(index=["per_position", "arm", "n"], values=[f"eta_comb_{K}", "eta_comb_1"],
                          aggfunc=["mean", "std"])
print(f"eta_comb over the four planes (4x4 is the published scale):")
print(view.round(4).to_string())

## 11. The decision

Tonight's ladder at four frames per position, Winsorized, 4x4, mean over planes, against `25`'s
`eta_comb_registered` at the same N. The ratio's uncertainty combines tonight's spread over planes
with `25`'s published one, in quadrature; it is printed beside the verdict and does not change it.
Then the same-night control, which compares the two cadences with nothing else moved.

In [ ]:
four = (ladder[(ladder.per_position == CADENCE) & (ladder.arm == "winsorized")]
        .groupby("n")[f"eta_comb_{K}"].agg(["mean", "std"]))
ref_v = stack06["eta_comb_registered"]["value"]
ref_u = stack06["eta_comb_registered"]["uncertainty"]
cmp = pd.DataFrame([{"n": n, "tonight": four.loc[n, "mean"], "every_frame_06": ref_v[str(n)],
                     "ratio": four.loc[n, "mean"] / ref_v[str(n)],
                     "ratio_u": four.loc[n, "mean"] / ref_v[str(n)] * np.hypot(
                         four.loc[n, "std"] / four.loc[n, "mean"], ref_u[str(n)] / ref_v[str(n)])}
                    for n in MATCHED]).set_index("n")
cmp["above"] = cmp.ratio >= THRESH
KEEP = bool(cmp.above.all())
print(cmp.round(4).to_string())
print(f"\nthreshold {THRESH} ({REGIME} download); break-even from tonight's own dead times "
      f"{1 / snr_gain:.4f}")
print("VERDICT:", "KEEP dithering every 4th frame" if KEEP else
      "DROP it -- every-frame dithering was right, and the saving was never real")

ctrl = (ladder[(ladder.per_position == 1)].groupby("n")[f"eta_comb_{K}"].agg(["mean", "std"]))
same = pd.DataFrame({"four_per_position": four["mean"], "one_per_position": ctrl["mean"]}).dropna()
same["ratio"] = same.four_per_position / same.one_per_position
print("\nsame-night control, 4 per position over 1 per position at the same N:")
print(same.round(4).to_string())
beyond = four[four.index > max(MATCHED)]
print("\nabove N = 18 tonight has the only ladder:", beyond.round(4).to_dict("index"))

## 12. Star clipping

`29`'s method, run on tonight. Nothing is registered - resampling smooths a peak, which is the one
number measured - so each raw sub is lined up with the reference by whole-pixel shifts per tile.
The deep image is the mean of every usable sub, greens averaged; the list is its isolated stars.

**Following the sky.** Each list's stars go through both nights' plate solves, and a star is kept
only if it sits at least 64 plane pixels inside *both* nights' frames. Session 06's cell B is read
from `star_peaks.csv`, cut the same way, so the two fractions are over the same patch of sky.
**Focus moved twice tonight**, so the fraction is also given per focus block: the spread inside one
night is the yardstick for the difference between two.

In [ ]:
ref_g1 = planes_of(REF.file)["G1"]
t0 = time.time()
offsets = {f: spatial.tile_offsets(ref_g1, planes_of(f)["G1"], TILES) for f in good.file}
spread = pd.Series({f: int(np.ptp(o.reshape(-1, 2), axis=0).max()) for f, o in offsets.items()})
MARGIN = int(max(np.abs(o).max() for o in offsets.values()))
print(f"offsets in {time.time() - t0:.0f} s; largest {MARGIN} px, so the margin")
print("spread between one frame's tiles, px:", spread.value_counts().sort_index().to_dict())
assert MARGIN + RADIUS <= BORDER, "the border must clear the realignment margin"

deep = None
for f in good.file:
    pl = planes_of(f)
    g = (spatial.realign(pl["G1"], offsets[f], MARGIN).astype(np.float64)
         + spatial.realign(pl["G2"], offsets[f], MARGIN)) / 2
    deep = g if deep is None else deep + g
deep /= len(good)
STARS = spatial.stars(deep, **DETECT)
print(f"deep image {deep.shape}; {len(STARS)} isolated stars")

In [ ]:
PH, PW = 1080, 1920


def to_world(wcs, rows, cols):
    return wcs.all_pix2world(2 * np.asarray(cols) + 0.5, 2 * np.asarray(rows) + 0.5, 0)


def to_plane(wcs, ra, dec):
    x, y = wcs.all_world2pix(ra, dec, 0)
    return (y - 0.5) / 2, (x - 0.5) / 2


def inside(r, c):
    r, c = np.asarray(r), np.asarray(c)
    return (r >= BORDER) & (r < PH - BORDER) & (c >= BORDER) & (c < PW - BORDER)


r08, c08 = STARS[:, 0] + MARGIN, STARS[:, 1] + MARGIN
KEEP08 = inside(r08, c08) & inside(*to_plane(w06, *to_world(w08, r08, c08)))

peaks06 = pd.read_csv(RESULTS / "star_peaks.csv")
b06 = peaks06[peaks06.cell == "B"].copy()
xy06 = b06.drop_duplicates("star").set_index("star")[["row", "col"]]
assert xy06.row.min() <= BORDER and xy06.col.min() <= BORDER, "29's margin, not the border, sets 06's cut"
keep06 = inside(xy06.row, xy06.col) & inside(*to_plane(w08, *to_world(w06, xy06.row, xy06.col)))
KEEP06 = set(xy06.index[keep06])
print(f"stars on the shared patch: tonight {int(KEEP08.sum())} of {len(STARS)}, "
      f"session 06 {len(KEEP06)} of {len(xy06)}")

In [ ]:
# Only the shared patch is read.  Its border clears the realignment margin and
# the peak box, which is what keeps every box on the image.
SHARED, SHARED_ID = STARS[KEEP08], np.flatnonzero(KEEP08)
t0 = time.time()
rows = []
for r in good.itertuples():
    pl = planes_of(r.file)
    for plane in PLANES:
        pk = stats.peaks(spatial.realign(pl[plane], offsets[r.file], MARGIN), SHARED, RADIUS)
        rows.append(pd.DataFrame({"star": SHARED_ID, "plane": plane,
                                  "focuspos": r.focuspos, "peak": pk}))
subs = pd.concat(rows, ignore_index=True)
print(f"{len(subs)} star-subs on the shared patch in {time.time() - t0:.0f} s")


def per_star(df):
    per = df.groupby(["star", "plane"]).peak.median().rename("peak_median").reset_index()
    per["clipped"] = per.peak_median >= FULL_SCALE
    per["clipped_safe"] = per.peak_median >= CEILING
    return per


def fractions(per):
    out = {}
    for col in ("clipped", "clipped_safe"):
        by = per.groupby("plane")[col].mean().to_dict()
        by["any"] = float(per.groupby("star")[col].any().mean())
        out[col] = by
    return out


per08 = per_star(subs)
b06["clipped_safe"] = b06.peak_median >= CEILING
per06 = b06[b06.star.isin(KEEP06)][["star", "plane", "peak_median", "clipped", "clipped_safe"]]
full06_any = float(b06.groupby("star").clipped.any().mean())
CLIP = {"tonight": fractions(per08), "session06_B": fractions(per06)}
print(f"reading check: session 06 cell B, whole list, any plane at {FULL_SCALE}: "
      f"{100 * full06_any:.2f}% of {b06.star.nunique()} (the protocol quotes 1.15% of 8033)")
show = pd.DataFrame({f"{who} {col}": 100 * pd.Series(v) for who, d in CLIP.items()
                     for col, v in d.items()}).T
print("\n% of stars clipped, on the shared patch (clipped = at 4095; clipped_safe = at "
      f"linear_to_at_least, {CEILING:.0f}):")
print(show[list(PLANES) + ["any"]].round(3).to_string())

BLOCKS = {int(fp): fractions(per_star(subs[subs.focuspos == fp]))["clipped"]["any"]
          for fp in sorted(subs.focuspos.unique(), reverse=True)}
print("\nany plane at 4095, per focus block (median over that block's subs):",
      {k: f"{100 * v:.2f}%" for k, v in BLOCKS.items()},
      "subs per block:", good.groupby("focuspos").size().to_dict())

## 13. Publish

`cadence_frames.csv` holds every light: its gap, whether a dither came before it (from the gaps and
from the log), its position and twin, its temperature, sky and verdict. `cadence_constants.json`
holds the constants, each with its provenance. The eta tables are the **Winsorized arm at 4x4**,
averaged over planes, because a model table is one curve; the spread over planes is the
uncertainty.

In [ ]:
measured_on = str(night.t.min().date())


def constant(value, unit, uncertainty, n, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty, "source_frames": int(n),
            "measured_on": measured_on, "notebook": "35_cadence.ipynb", "note": note}


def r6(v):
    return None if v is None or not np.isfinite(v) else round(float(v), 6)


def table(per_position, arm):
    g = ladder[(ladder.per_position == per_position) & (ladder.arm == arm)].groupby("n")[f"eta_comb_{K}"]
    return {str(n): r6(v) for n, v in g.mean().items()}, {str(n): r6(v) for n, v in g.std().items()}


SETTINGS = (f"Winsorized sigma clip {REJECTION['winsorized']['sigma_low']}/"
            f"{REJECTION['winsorized']['sigma_high']}, average, additive normalisation, equal "
            f"weights, Lanczos-3 registration of each Bayer plane separately (SplitCFA, no "
            f"debayer), {K}x{K} binning, signal ROI {SIGNAL_ROI} in plane pixels of tonight's "
            f"reference, moved onto session 06's sky.  Twins split by dither position, so a twin "
            f"pair shares no position; the ideal is the across-position single frame")
TIMING = (f"gap in front of a frame = its DATE-OBS minus the previous one's, minus {EXPTIME:.0f} s.  "
          f"Dither gaps are those over {BARE_FACTOR:g}x the night's median gap, checked against "
          f"the cadence and the ASIAIR log (disagreements: {off_cadence or 'none'} / "
          f"{disagree or 'none'}).  Interruptions over {INTERRUPTION_FACTOR:g}x the median "
          f"dither gap are excluded: frames {night.n[night.interruption].tolist()} (the flip and "
          f"two autofocus runs, each also holding a dither)")
v4, u4 = table(CADENCE, "winsorized")
v4n, u4n = table(CADENCE, "none")
v1, u1 = table(1, "winsorized")
n_ladder = len(in_ladder)

constants = {
    "download_s": constant(r6(download), "s", r6(download_se), len(bare),
        "rule 4: mean gap over frames with no dither in front.  Measured, where session 06 could "
        "only bound it.  " + TIMING),
    "dither_settle_s": constant(r6(settle), "s", r6(settle_se), len(dith),
        f"rule 4: mean dither gap minus the download.  The ASIAIR log's own settle times give "
        f"{settle_log:.2f} s over {len(settles)} dithers at 1 s resolution, "
        f"{int(settles.timeout.sum())} of them a timeout.  " + TIMING),
    "t_dead": constant(r6(t_dead), "s", r6(t_dead_se), len(steady),
        f"[the model's t_dead at one dither every {CADENCE} frames] download + settle / {CADENCE}: "
        f"a clean night at this cadence.  Chosen over rule 4's night mean (t_dead_rule4) with "
        f"Denis, 2026-10-06, because the excluded interruptions each hold a dither and leave the "
        f"night under one in {CADENCE}.  " + TIMING),
    "t_dead_rule4": constant(r6(t_dead_rule4), "s", r6(t_dead_rule4_se), len(steady),
        f"rule 4 as written: mean of every gap, interruptions excluded.  {dither_share:.4f} of "
        f"these gaps are dithers, against 1/{CADENCE}.  " + TIMING),
    "t_dead_every_frame": constant(r6(t_dead_every), "s", r6(settle_se), len(steady),
        "download + settle: what this rig costs per sub dithering every frame, from tonight's own "
        "split.  Beside sky_constants t_dead (session 06, measured that way directly)"),
    "dither_cadence": constant(f"every {CADENCE} frames", "frames per dither", None, len(night),
        "as the ASIAIR was set, and as the gaps and the log both show"),
    "eta_comb_cadence": constant(v4, "measured sd reduction against the ideal sqrt(N) of a "
        "registered sub", u4, n_ladder,
        f"rule 5 at {CADENCE} frames per dither position, keyed by N; {len(LADDER_POS)} positions "
        f"in all, a stack of N holds N/{CADENCE}.  The top rung is the half-stack, {HALF}: two "
        f"disjoint stacks of 64 need 128 frames and the flip left {len(good)}.  {SETTINGS}.  "
        "Uncertainty is the spread over the four planes"),
    "eta_comb_cadence_no_rejection": constant(v4n, "measured sd reduction against the ideal "
        "sqrt(N) of a registered sub", u4n, n_ladder,
        "the same ladder with rejection off.  " + SETTINGS),
    "eta_comb_one_per_position": constant(v1, "measured sd reduction against the ideal sqrt(N) "
        "of a registered sub", u1, n_ladder,
        "the same-night control: one frame per position, so a stack dithered every frame, from "
        "tonight.  A check on the cross-night comparison, not the verdict.  " + SETTINGS),
    "eta_ratio_vs_every_frame": constant(
        {str(n): r6(r.ratio) for n, r in cmp.iterrows()},
        "eta_comb_cadence / stack_constants eta_comb_registered (session 06), at matched N",
        {str(n): r6(r.ratio_u) for n, r in cmp.iterrows()}, n_ladder,
        "the protocol's decision quantity.  Across nights and setpoints (-10 C tonight, -20 C "
        "then); the same-night control is eta_comb_one_per_position.  Uncertainty combines "
        "tonight's spread over planes with 25's published one"),
    "cadence_decision": constant(
        {"verdict": "keep every-4th-frame dithering" if KEEP else "drop it; dither every frame",
         "threshold": THRESH, "download_regime": REGIME,
         "break_even_measured": r6(1 / snr_gain), "snr_gain_if_no_loss_pct": r6(100 * (snr_gain - 1)),
         "rungs_above": [int(n) for n in cmp.index[cmp.above]]},
        "verdict and its inputs", None, n_ladder,
        f"protocols/08-dither-cadence.md, fixed before the data: keep if the ratio is at or above "
        f"{THRESHOLD['fast']} (fast download) or {THRESHOLD['slow']} (slow) at every matched "
        f"rung {MATCHED}.  break_even_measured is 1 / sqrt((t + t_dead_every_frame) / (t + t_dead)) "
        f"at t = {EXPTIME:.0f} s, from tonight's own dead times, given for comparison only"),
    "position_locked_share": constant(
        {p: {**{str(k): r6(one.loc[p, (f"locked_{k}", "")]) for k in SCALES},
             f"{K}_star_masked": r6(one.loc[p, ("locked_masked", "")])} for p in PLANES},
        "share of a registered single frame's noise that cancels between frames at one dither "
        "position, keyed by bin size", None, n_ladder,
        "sqrt(across^2 - within^2) / across: pairs at different dither positions against pairs "
        "at the same one, in the signal ROI.  What a dither has to beat down.  NOT the sensor's "
        "fixed pattern alone: frames at one position share their registration shift, so star "
        "residuals of the resampling cancel within a position too, and within-position pairs are "
        "2 minutes apart against about 9.  The star-masked figure (27's tight mask, "
        f"{MASK_KW}, covering {({p: round(v, 3) for p, v in MASK_SHARE.items()})} of the ROI) "
        "points at how much is stars.  A diagnostic: the ladder is unmasked, as 25's was"),
    "clipped_star_fraction": constant(
        {who: {col: {p: r6(v) for p, v in d.items()} for col, d in w.items()}
         for who, w in CLIP.items()},
        "fraction of the star list whose median peak reaches the level, per plane and any plane",
        None, len(good),
        f"rule 6: 29's method on tonight's {len(good)} usable subs, against session 06's cell B "
        f"(same gain and exposure) read from star_peaks.csv.  Only stars at least {BORDER} plane px "
        f"inside both nights' frames: {int(KEEP08.sum())} tonight, {len(KEEP06)} then.  'clipped' "
        f"is at {FULL_SCALE}; 'clipped_safe' at linear_to_at_least, {CEILING:.1f}.  Not pass/fail: "
        f"the difference is the night-to-night spread.  Across setpoints, -10 C and -20 C, where "
        f"the dark term is negligible at both"),
    "clipped_star_fraction_by_focus": constant(
        {str(k): r6(v) for k, v in BLOCKS.items()},
        "fraction of tonight's shared-patch list clipped at 4095 in any plane, per focus position",
        None, len(good),
        "the median is taken over each block's own subs.  The spread inside one night, as a "
        "yardstick for the spread between two"),
    "F_sky": constant({p: r6(F_SKY[p]) for p in PLANES}, "e-/px/s",
        {p: r6(F_SKY_PED_ERR) for p in PLANES}, len(good),
        f"session 06's rule 3: modal level of the sky ROI {SKY_MOSAIC} (mosaic px, moved onto "
        f"session 06's sky), minus the bracket pedestal, x g = {G} (-10 C), / {EXPTIME:.0f} s, "
        f"averaged over usable frames.  Uncertainty is the pedestal's: the scatter inside the "
        f"populated offset state and half the {STATE_SEP:.3f}-count step to the other, which a "
        f"light cannot be assigned to on its own.  A by-product"),
    "F_sky_green": constant(r6(F_SKY["green"]), "e-/px/s", r6(F_SKY_SD["green"]), len(good),
        "mean of G1 and G2.  Uncertainty is the frame-to-frame scatter across the night"),
    "pedestal_brackets": constant({p: r6(v) for p, v in PEDESTAL.items()}, "ADC counts",
        r6(PEDESTAL_U), int(near.sum()),
        f"rule 2: per-plane mean of the 20 + 20 bracket frames in the populated offset state; "
        f"{int(state['far'].sum())} sat in a second state {STATE_SEP:.3f} counts away.  "
        f"Uncertainty: sd inside the state ({PEDESTAL_SD:.3f}) and half the step, in quadrature.  "
        f"bias_sweep.csv has {published:.3f} at this gain and offset"),
    "white_balance_gate": constant(GATE1, "boolean: modal value step 16 on R, G1, G2, B",
        None, len(brackets), "gate 1, on every bracket frame, in stored units"),
    "frames": constant(
        {"lights": len(night), "before_flip": int(night.flipped.sum()),
         "rejected": int(night.rejected.sum()), "usable": len(good), "in_ladder": n_ladder,
         "temp_flagged": int(night.temp_flag.sum())},
        "frames", None, len(night),
        "before_flip: the protocol's transit time was wrong by an hour and the flip came after "
        "frame 28 (D114).  Timing uses every light; pixel work only the post-flip ones"),
    "roi": constant(
        {"signal_roi": list(SIGNAL_MOSAIC), "sky_roi": list(SKY_MOSAIC), "turn_deg": r6(TURN),
         "reference": REF.file, "reference_session06": REF06.file},
        "x, y, w, h in mosaic pixels of tonight's reference", None, 1,
        "session 06's ROIs followed onto the same sky through both references' plate solves "
        "(agreed with Denis, 2026-10-06).  The turn is not followed; its effect on a corner is "
        "printed in section 3"),
}

cols = (["n", "file", "t", "gap_s", "dither", "dither_log", "interruption", "position", "twin",
         "rotator", "flipped", "focuspos", "ccd_temp", "temp_off", "temp_flag"]
        + [f"sky_mode_{p}" for p in PLANES] + ["sky_mode_green", "sky_dev", "rejected"]
        + [f"F_sky_{p}" for p in PLANES] + ["F_sky_green"])
night[cols].to_csv(FRAMES_CSV, index=False, float_format="%.6f")
with open(CONSTANTS, "w") as f:
    json.dump(constants, f, indent=2)
M.Constants.load(CONSTANTS)       # the provenance gate, before anything else reads it
print("wrote", FRAMES_CSV.name, f"({len(night)} rows),", CONSTANTS.name)

## 14. Clean up

The registered planes and stacks are disposable and several GB. They are removed only once the two
files above exist.

In [ ]:
assert CONSTANTS.exists() and FRAMES_CSV.exists()
shutil.rmtree(TEMP, ignore_errors=True)
print("removed", TEMP)